Qwen 2.5 Instruct Model: https://huggingface.co/Qwen/Qwen3.5-397B-A17B#benchmark-results:~:text=post%20Qwen3.5.-,Model%20Overview,-Type%3A%20Causal%20Language

In [1]:
!pip -q install -U "transformers>=4.41.0" accelerate bitsandbytes sentencepiece pandas scikit-learn

import json, time
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


Load Qwen2.5

In [2]:
model_id = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
    padding_side="left"
)

tokenizer.pad_token = tokenizer.eos_token


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)

print("Loaded:", model_id)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-7B-Instruct


Load test set (kaggle)

In [5]:
KAGGLE_CLEAN_PATH = "../kaggle_data.json"

with open(KAGGLE_CLEAN_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

X_test = [item["review_content"] for item in test_data]
y_test = [1 if item["is_positive"] else 0 for item in test_data]

print("Test samples (Kaggle):", len(X_test))
print("Pos:", sum(y_test), "Neg:", len(y_test)-sum(y_test))


Test samples (Kaggle): 117458
Pos: 55789 Neg: 61669


Qwen classifier with strict label output to match our custom models

In [6]:
def qwen_predict_labels(texts, batch_size=16, max_new_tokens=3, max_length=1024):
    preds = []
    t0 = time.time()

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        prompts = []

        for txt in batch:
            messages = [
                {"role": "system", "content":
                 "You are a strict sentiment classifier for game reviews. "
                 "Output ONLY one word: Positive or Negative."},
                {"role": "user", "content": f"Review:\n{txt}\n\nLabel:"}
            ]
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # deterministic for benchmarking
                temperature=0.0,
                eos_token_id=tokenizer.eos_token_id,
            )

        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)

        for full in decoded:
            tail = full.split("Label:")[-1].strip().lower()
            if tail.startswith("positive"):
                preds.append(1)
            elif tail.startswith("negative"):
                preds.append(0)
            else:
                preds.append(0)  # fallback

    t1 = time.time()
    return preds, (t1 - t0)


In [ ]:
N = 5000
y_pred, seconds = qwen_predict_labels(X_test[:N], batch_size=16)

acc = accuracy_score(y_test[:N], y_pred)
prec = precision_score(y_test[:N], y_pred, zero_division=0)
rec = recall_score(y_test[:N], y_pred, zero_division=0)
f1 = f1_score(y_test[:N], y_pred, zero_division=0)
cm = confusion_matrix(y_test[:N], y_pred)

print("="*50)
print("QWEN EVALUATION ON KAGGLE TEST SET")
print("="*50)
print(f"N:         {N}")
print(f"Seconds:   {seconds:.2f}  (sec/ex: {seconds/N:.4f})")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("\n--- Confusion Matrix ---")
print(cm)

print("\n--- Classification Report ---")
print(classification_report(y_test[:N], y_pred, target_names=["Negative","Positive"], zero_division=0))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
